# Machine Learning Assignment 2 — Training Notebook
**Predicting online shopper purchase intent**  
M.Tech AIML/DSE — BITS Pilani (WILP)

This notebook trains the five classifiers required by the assignment on the UCI Online Shoppers Purchasing Intention dataset, evaluates them on a stratified hold-out set, and saves sklearn pipelines for the Streamlit app.

**BITS Virtual Lab:** clone or upload the GitHub repo, open this notebook, Run All. If `online_shoppers_intention.csv` is not on disk, the next cell downloads it from GitHub automatically.

In [ ]:
from pathlib import Path
import json
import urllib.request

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATASET_URL = (
    "https://raw.githubusercontent.com/hemant935/shopper-purchase-intent/"
    "main/online_shoppers_intention.csv"
)

cwd = Path.cwd()
candidates = [cwd, cwd.parent, Path("/content"), Path.home() / "work"]
PROJECT_ROOT = next((p for p in candidates if (p / "online_shoppers_intention.csv").exists()), cwd)
SESSION_CSV = PROJECT_ROOT / "online_shoppers_intention.csv"
MODEL_DIR = PROJECT_ROOT / "model"
HOLDOUT_CSV = PROJECT_ROOT / "test_data.csv"
MODEL_DIR.mkdir(exist_ok=True)

if not SESSION_CSV.exists():
    print("Local CSV not found — downloading from GitHub...")
    urllib.request.urlretrieve(DATASET_URL, SESSION_CSV)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", SESSION_CSV)
print("Dataset exists:", SESSION_CSV.exists())

## 1. Load and inspect the dataset

In [ ]:
shopper_sessions = pd.read_csv(SESSION_CSV)

print("Shape:", shopper_sessions.shape)
print("Columns:", list(shopper_sessions.columns))
print("Missing values:", int(shopper_sessions.isna().sum().sum()))
print("\nClass balance (Revenue):")
print(shopper_sessions["Revenue"].value_counts())
print(shopper_sessions["Revenue"].value_counts(normalize=True).round(4))
print("\nMonth levels:", sorted(shopper_sessions["Month"].unique().tolist()))
print("VisitorType levels:", shopper_sessions["VisitorType"].unique().tolist())
shopper_sessions.head()

## 2. Encode labels, split, and save `test_data.csv`

`Weekend` and `Revenue` are mapped to 0/1. The split is **stratified** on the purchase label so the 15.5% positive class is preserved in both sides. The raw hold-out rows (original columns, including `Revenue`) are written to `test_data.csv` for the Streamlit app.

In [ ]:
NUMERIC_SESSION_COLS = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
    "OperatingSystems", "Browser", "Region", "TrafficType",
    "Weekend",
]
CATEGORY_SESSION_COLS = ["Month", "VisitorType"]

session_features = shopper_sessions.drop(columns=["Revenue"]).copy()
session_features["Weekend"] = session_features["Weekend"].astype(int)
purchase_label = shopper_sessions["Revenue"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    session_features,
    purchase_label,
    test_size=0.20,
    stratify=purchase_label,
    random_state=42,
)

shopper_sessions.loc[X_test.index].to_csv(HOLDOUT_CSV, index=False)
print(f"Train: {X_train.shape} | Hold-out: {X_test.shape}")
print("Hold-out positive rate:", round(float(y_test.mean()), 4))
print("Saved", HOLDOUT_CSV)

## 3. Shared preprocessing and the five classifiers

Numeric columns (including integer-coded OS / browser / region / traffic type) are standardised. `Month` and `VisitorType` are one-hot encoded. Each saved artifact is a full `Pipeline` so the Streamlit app applies the same transform at inference time.

`class_weight='balanced'` is used where the estimator supports it (Logistic Regression, Decision Tree, Random Forest) because of the 15.5% purchase rate.

In [ ]:
def build_session_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("numeric_scale", StandardScaler(), NUMERIC_SESSION_COLS),
            (
                "category_onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                CATEGORY_SESSION_COLS,
            ),
        ]
    )


def score_holdout(y_true, y_pred, y_proba):
    return {
        "Accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "AUC": round(float(roc_auc_score(y_true, y_proba)), 4),
        "Precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "Recall": round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "F1": round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        "MCC": round(float(matthews_corrcoef(y_true, y_pred)), 4),
    }


intent_models = {
    "Logistic Regression": {
        "file": "logistic_regression.pkl",
        "estimator": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    },
    "Decision Tree": {
        "file": "decision_tree.pkl",
        "estimator": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    },
    "kNN": {
        "file": "knn.pkl",
        "estimator": KNeighborsClassifier(n_neighbors=5),
    },
    "Naive Bayes": {
        "file": "naive_bayes.pkl",
        "estimator": GaussianNB(),
    },
    "Random Forest (Ensemble)": {
        "file": "random_forest.pkl",
        "estimator": RandomForestClassifier(
            n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
        ),
    },
}

## 4. Train, evaluate, and persist pipelines

In [ ]:
holdout_metrics = {}
rf_pipeline = None

for model_name, spec in intent_models.items():
    pipeline = Pipeline(
        steps=[
            ("session_prep", build_session_preprocessor()),
            ("classifier", spec["estimator"]),
        ]
    )
    pipeline.fit(X_train, y_train)
    predicted = pipeline.predict(X_test)
    purchase_proba = pipeline.predict_proba(X_test)[:, 1]
    holdout_metrics[model_name] = score_holdout(y_test, predicted, purchase_proba)
    joblib.dump(pipeline, MODEL_DIR / spec["file"])
    print("Saved", model_name)
    if model_name == "Random Forest (Ensemble)":
        rf_pipeline = pipeline

(MODEL_DIR / "metrics.json").write_text(json.dumps(holdout_metrics, indent=2))
print("Wrote", MODEL_DIR / "metrics.json")

## 5. Hold-out comparison table (screenshot this cell on BITS Virtual Lab)

In [ ]:
print("BITS Virtual Lab — Assignment 2 execution proof")
print("Dataset:", SESSION_CSV.name, shopper_sessions.shape)
print("Hold-out rows:", len(X_test), "| positive rate:", round(float(y_test.mean()), 4))
print()
metrics_table = pd.DataFrame(holdout_metrics).T[
    ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
]
print(metrics_table.to_string())
metrics_table

## 6. Random Forest feature importances (for README observations)

In [ ]:
prepared = rf_pipeline.named_steps["session_prep"]
forest = rf_pipeline.named_steps["classifier"]
importance = pd.Series(
    forest.feature_importances_, index=prepared.get_feature_names_out()
).sort_values(ascending=False)
print(importance.head(8).to_string())